# 03 - Anomaly Detection (Autoencoder)

Train a Dense Autoencoder to detect unusual spending patterns.

**Architecture**: Dense(32) → Dense(16) → Dense(8) [bottleneck] → Dense(16) → Dense(32) → Dense(input_dim)

**Key idea**: Train on *normal* transactions only. At inference, transactions with high reconstruction error are anomalies.

**Pipeline**:
1. Feature engineering from transaction data
2. Train autoencoder on normal transactions
3. Find optimal anomaly threshold
4. Compare with Isolation Forest baseline
5. Export to TFLite + threshold config for Flutter

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, roc_curve

from preprocessing import APP_CATEGORIES, CATEGORY_TO_IDX
from export_tflite import export_keras_to_tflite, export_anomaly_params, verify_tflite_model

print(f'TensorFlow version: {tf.__version__}')

## 1. Load & Prepare Data

In [ ]:
# Load the anomaly training data (preprocessed in notebook 01)
df = pd.read_csv('../data/processed/anomaly_train.csv')
print(f'Dataset: {len(df)} rows')
print(f'Anomaly distribution:')
print(df['is_anomaly'].value_counts())
print(f'\nAnomaly rate: {df["is_anomaly"].mean():.2%}')

In [ ]:
# Feature engineering for autoencoder
# We use features available at transaction time in the Flutter app

def engineer_features(df):
    """Create features for anomaly detection."""
    features = pd.DataFrame()
    
    # Amount (log-transformed for better distribution)
    features['log_amount'] = np.log1p(df['amount'].astype(float))
    
    # Category one-hot encoding
    if 'category' in df.columns:
        for cat in APP_CATEGORIES:
            features[f'cat_{cat}'] = (df['category'] == cat).astype(float)
    elif 'category_idx' in df.columns:
        for idx, cat in enumerate(APP_CATEGORIES):
            features[f'cat_{cat}'] = (df['category_idx'] == idx).astype(float)
    
    # Day of week (cyclical encoding)
    if 'date' in df.columns:
        dates = pd.to_datetime(df['date'])
        dow = dates.dt.dayofweek
        features['dow_sin'] = np.sin(2 * np.pi * dow / 7)
        features['dow_cos'] = np.cos(2 * np.pi * dow / 7)
        
        # Day of month (cyclical)
        dom = dates.dt.day
        features['dom_sin'] = np.sin(2 * np.pi * dom / 31)
        features['dom_cos'] = np.cos(2 * np.pi * dom / 31)
        
        # Hour (if available, otherwise 0)
        hour = dates.dt.hour
        features['hour_sin'] = np.sin(2 * np.pi * hour / 24)
        features['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    
    # Transaction type
    if 'type' in df.columns:
        features['is_expense'] = (df['type'] == 'expense').astype(float)
    
    return features

features = engineer_features(df)
print(f'Feature matrix shape: {features.shape}')
print(f'Features: {list(features.columns)}')

In [ ]:
# Split: train on normal transactions only, test on all
is_anomaly = df['is_anomaly'].values.astype(bool)

X_normal = features[~is_anomaly].values
X_anomaly = features[is_anomaly].values

print(f'Normal transactions: {len(X_normal)}')
print(f'Anomaly transactions: {len(X_anomaly)}')

# Scale features
scaler = StandardScaler()
X_normal_scaled = scaler.fit_transform(X_normal)
X_anomaly_scaled = scaler.transform(X_anomaly)

# Train/validation split (from normal data only)
np.random.seed(42)
n_val = int(len(X_normal_scaled) * 0.15)
indices = np.random.permutation(len(X_normal_scaled))
X_train = X_normal_scaled[indices[n_val:]]
X_val = X_normal_scaled[indices[:n_val]]

print(f'\nTrain: {len(X_train)} samples')
print(f'Validation: {len(X_val)} samples')
print(f'Test (anomalies): {len(X_anomaly_scaled)} samples')

## 2. Build Autoencoder

In [ ]:
INPUT_DIM = X_train.shape[1]

# Encoder
encoder_input = tf.keras.Input(shape=(INPUT_DIM,))
x = tf.keras.layers.Dense(32, activation='relu')(encoder_input)
x = tf.keras.layers.Dense(16, activation='relu')(x)
bottleneck = tf.keras.layers.Dense(8, activation='relu', name='bottleneck')(x)

# Decoder
x = tf.keras.layers.Dense(16, activation='relu')(bottleneck)
x = tf.keras.layers.Dense(32, activation='relu')(x)
decoder_output = tf.keras.layers.Dense(INPUT_DIM, activation='linear')(x)

autoencoder = tf.keras.Model(encoder_input, decoder_output, name='anomaly_autoencoder')

autoencoder.compile(
    optimizer='adam',
    loss='mse',
)

autoencoder.summary()

## 3. Train Autoencoder

In [ ]:
history = autoencoder.fit(
    X_train, X_train,  # Input == Target for autoencoder
    validation_data=(X_val, X_val),
    epochs=100,
    batch_size=32,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=10, restore_best_weights=True, monitor='val_loss'
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            factor=0.5, patience=5, monitor='val_loss'
        ),
    ],
)

In [ ]:
# Plot training curves
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history.history['loss'], label='Train Loss')
ax.plot(history.history['val_loss'], label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Autoencoder Training')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig('../models/saved/anomaly_training_curves.png', dpi=100)
plt.show()

## 4. Find Optimal Threshold

In [ ]:
# Compute reconstruction error for normal and anomaly data
def compute_reconstruction_error(model, X):
    """MSE per sample between input and reconstruction."""
    X_pred = model.predict(X, verbose=0)
    return np.mean((X - X_pred) ** 2, axis=1)

errors_normal_train = compute_reconstruction_error(autoencoder, X_train)
errors_normal_val = compute_reconstruction_error(autoencoder, X_val)
errors_anomaly = compute_reconstruction_error(autoencoder, X_anomaly_scaled)

print(f'Normal train  - mean: {errors_normal_train.mean():.6f}, std: {errors_normal_train.std():.6f}')
print(f'Normal val    - mean: {errors_normal_val.mean():.6f}, std: {errors_normal_val.std():.6f}')
print(f'Anomaly       - mean: {errors_anomaly.mean():.6f}, std: {errors_anomaly.std():.6f}')
print(f'\nSeparation ratio: {errors_anomaly.mean() / errors_normal_val.mean():.1f}x')

In [ ]:
# Plot reconstruction error distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax = axes[0]
ax.hist(errors_normal_val, bins=50, alpha=0.7, label='Normal', density=True)
ax.hist(errors_anomaly, bins=50, alpha=0.7, label='Anomaly', density=True)
ax.set_xlabel('Reconstruction Error (MSE)')
ax.set_ylabel('Density')
ax.set_title('Reconstruction Error Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# ROC curve
ax = axes[1]
all_errors = np.concatenate([errors_normal_val, errors_anomaly])
all_labels = np.concatenate([np.zeros(len(errors_normal_val)), np.ones(len(errors_anomaly))])
fpr, tpr, thresholds_roc = roc_curve(all_labels, all_errors)
auc = roc_auc_score(all_labels, all_errors)
ax.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig('../models/saved/anomaly_error_distribution.png', dpi=100)
plt.show()

print(f'AUC-ROC: {auc:.4f}')

In [ ]:
# Find threshold that maximizes F1 score
best_f1 = 0
best_threshold = 0

# Search over percentiles of normal data errors
for percentile in range(90, 100):
    threshold = np.percentile(errors_normal_val, percentile)
    y_pred = (all_errors > threshold).astype(int)
    f1 = f1_score(all_labels, y_pred)
    prec = precision_score(all_labels, y_pred)
    rec = recall_score(all_labels, y_pred)
    print(f'  P{percentile}: threshold={threshold:.6f}, precision={prec:.3f}, recall={rec:.3f}, F1={f1:.3f}')
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print(f'\nBest threshold: {best_threshold:.6f} (F1: {best_f1:.3f})')

# Also try mean + k*std approach
for k in [2, 2.5, 3, 3.5]:
    threshold_k = errors_normal_train.mean() + k * errors_normal_train.std()
    y_pred_k = (all_errors > threshold_k).astype(int)
    f1_k = f1_score(all_labels, y_pred_k)
    prec_k = precision_score(all_labels, y_pred_k)
    rec_k = recall_score(all_labels, y_pred_k)
    print(f'  mean+{k}σ: threshold={threshold_k:.6f}, precision={prec_k:.3f}, recall={rec_k:.3f}, F1={f1_k:.3f}')

## 5. Baseline: Isolation Forest

In [ ]:
# Compare with Isolation Forest
iso_forest = IsolationForest(
    contamination=0.02,  # Expected anomaly rate
    random_state=42,
    n_estimators=200,
)
iso_forest.fit(X_normal_scaled)

# Predict on combined set
all_X = np.vstack([X_val, X_anomaly_scaled])
iso_pred = iso_forest.predict(all_X)
iso_labels = (iso_pred == -1).astype(int)  # -1 = anomaly in sklearn

iso_prec = precision_score(all_labels, iso_labels)
iso_rec = recall_score(all_labels, iso_labels)
iso_f1 = f1_score(all_labels, iso_labels)

print('Isolation Forest results:')
print(f'  Precision: {iso_prec:.3f}')
print(f'  Recall:    {iso_rec:.3f}')
print(f'  F1:        {iso_f1:.3f}')

print(f'\n--- Comparison ---')
print(f'Autoencoder F1: {best_f1:.3f}')
print(f'Isolation Forest F1: {iso_f1:.3f}')
print(f'Winner: {"Autoencoder" if best_f1 >= iso_f1 else "Isolation Forest"}')

## 6. Export to TFLite

In [ ]:
# Save full Keras model
autoencoder.save('../models/saved/anomaly_autoencoder_keras')
print('Keras model saved.')

# Export TFLite with quantization
tflite_meta = export_keras_to_tflite(
    autoencoder,
    output_path='../models/tflite/anomaly_detector.tflite',
    quantize=True,
)
print(f'\nTFLite model: {tflite_meta["size_kb"]:.1f} KB')

In [ ]:
import json

# Export scaler parameters (mean and std for each feature)
scaler_params = {
    'feature_names': list(features.columns),
    'mean': scaler.mean_.tolist(),
    'std': scaler.scale_.tolist(),
}

# Export anomaly detection config
anomaly_config = {
    'input_dim': INPUT_DIM,
    'feature_names': list(features.columns),
    'scaler': scaler_params,
    'threshold': float(best_threshold),
    'threshold_method': 'best_f1_percentile',
    'normal_error_mean': float(errors_normal_train.mean()),
    'normal_error_std': float(errors_normal_train.std()),
    'metrics': {
        'auc_roc': float(auc),
        'best_f1': float(best_f1),
    },
}

with open('../models/tflite/anomaly_config.json', 'w') as f:
    json.dump(anomaly_config, f, indent=2)
print('Anomaly config saved.')
print(f'\nConfig summary:')
print(f'  Input dim: {INPUT_DIM}')
print(f'  Threshold: {best_threshold:.6f}')
print(f'  AUC-ROC: {auc:.4f}')
print(f'  Best F1: {best_f1:.3f}')

In [ ]:
# Verify TFLite model
sample = X_val[:5].astype(np.float32)
tflite_output = verify_tflite_model('../models/tflite/anomaly_detector.tflite', sample)

keras_output = autoencoder.predict(sample, verbose=0)

print(f'\nKeras vs TFLite reconstruction comparison:')
for i in range(5):
    keras_err = np.mean((sample[i] - keras_output[i]) ** 2)
    tflite_err = np.mean((sample[i] - tflite_output[i]) ** 2)
    print(f'  Sample {i}: Keras error={keras_err:.6f}, TFLite error={tflite_err:.6f}')

In [ ]:
# Test on some examples
print(f'\n--- Sample Anomaly Scores ---')
print(f'{"Type":>10s}  {"Error":>10s}  {"Threshold":>10s}  {"Anomaly?":>10s}')
print('-' * 50)

for i in range(min(5, len(errors_normal_val))):
    err = errors_normal_val[i]
    is_anom = 'YES' if err > best_threshold else 'no'
    print(f'{"Normal":>10s}  {err:>10.6f}  {best_threshold:>10.6f}  {is_anom:>10s}')

for i in range(min(5, len(errors_anomaly))):
    err = errors_anomaly[i]
    is_anom = 'YES' if err > best_threshold else 'no'
    print(f'{"Anomaly":>10s}  {err:>10.6f}  {best_threshold:>10.6f}  {is_anom:>10s}')

## Results Summary

### Model Architecture
```
Input(N) → Dense(32, relu) → Dense(16, relu) → Dense(8, relu) [bottleneck]
         → Dense(16, relu) → Dense(32, relu) → Dense(N, linear)
```

### Files Exported for Flutter
| File | Description |
|------|-------------|
| `anomaly_detector.tflite` | Quantized autoencoder model |
| `anomaly_config.json` | Threshold, scaler params, feature names |

### Key Metrics
- Fill in after running: AUC-ROC, F1, precision, recall
- Anomaly = reconstruction error > threshold

### Next: Copy exports to Flutter
```bash
cp ../models/tflite/anomaly_detector.tflite ../../app/assets/models/
cp ../models/tflite/anomaly_config.json ../../app/assets/models/
```